In [ ]:
"""
# ========================================================
# 📘 VisoSpeak - 01_preprocessing.ipynb
# ========================================================

This notebook performs the first stage of the lip reading pipeline:
Detects faces and mouths from a silent video and saves cropped mouth images and metadata.

🔹 INPUT:
    - A silent speaking video file (MPG format)
    - Located at: data/raw_videos/{video_name}.mpg

🔹 OUTPUT:
    - Cropped mouth images (PNG format)
        ➤ Saved to: data/frames/{video_name}/mouth_0000.png, ...
    - Metadata JSON containing mouth crop coordinates
        ➤ Saved to: data/processed/{video_name}/crop_metadata_{video_name}.json

🔹 MAIN STEPS:
    1. Load the video
    2. Detect faces and mouths using Haar cascades
    3. Save the first valid mouth crop and all mouth regions
    4. Export bounding box coordinates to JSON for downstream use
"""
"""
# ========================================================
# ✅ STEP 1: IMPORTS & CONFIGURATION
# ========================================================
try:
    import cv2
    import matplotlib
    print(f"[✅] OpenCV {cv2.__version__}, Matplotlib {matplotlib.__version__} already installed.")
except ImportError:
    print("[⚠️] Installing missing dependencies...")
    !pip install opencv-python matplotlib
    import cv2
    import matplotlib


import os
import cv2
import json
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

# 📂 Absolute path to your project’s data folder
DATA_DIR = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data"

# Change this to your video name (without extension)
video_name = "vid_001"
video_file = f"{video_name}.mpg"
video_path = os.path.join(DATA_DIR, "raw_videos", video_file)

print(f"📥 Video Path: {video_path}")


# ========================================================
# ✅ STEP 2: DOWNLOAD HAAR CASCADES IF NEEDED
# ========================================================
def download_haar_cascades():
    print("\n🔧 Checking Haar cascade files...")
    haar_dir = cv2.data.haarcascades
    cascades = {
        "haarcascade_frontalface_default.xml":
        "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml",
        
        "haarcascade_mcs_mouth.xml":
        "https://raw.githubusercontent.com/atduskgreg/opencv-processing/master/lib/cascade-files/haarcascade_mcs_mouth.xml"
    }

    for file, url in cascades.items():
        path = os.path.join(haar_dir, file)
        if not os.path.exists(path):
            print(f"⬇️  Downloading {file}")
            urllib.request.urlretrieve(url, path)
        else:
            print(f"✅ {file} already exists.")

download_haar_cascades()

# ========================================================
# ✅ STEP 3: LOAD HAAR CLASSIFIERS
# ========================================================
print("\n🧠 Loading Haar Classifiers...")
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
mouth_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_mcs_mouth.xml")

assert not face_cascade.empty(), "❌ Failed to load face cascade"
assert not mouth_cascade.empty(), "❌ Failed to load mouth cascade"
print("✅ Haar classifiers loaded.")


# ========================================================
# ✅ STEP 4: PROCESS VIDEO FRAMES
# ========================================================
def process_video_frames(video_path, video_name):
    print(f"\n🎬 Processing: {video_path}")
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"❌ Video not found: {video_path}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"❌ Could not open video file: {video_path}")

    frame_index = 0
    saved_frame_count = 0
    frame_metadata = []

    frame_save_dir = os.path.join(DATA_DIR, "frames", video_name)
    os.makedirs(frame_save_dir, exist_ok=True)

    while True:
        ret, frame = cap.read()
        if not ret:
            print("✅ End of video.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

        if len(faces) == 0:
            print(f"🔍 Frame {frame_index}: No face detected.")
        else:
            for (x, y, w, h) in faces:
                roi_gray = gray[y + h // 2 : y + h, x:x + w]
                mouths = mouth_cascade.detectMultiScale(roi_gray, scaleFactor=1.3, minNeighbors=20, minSize=(50, 30))

                data = {
                    "frame_index": frame_index,
                    "face_coords": (int(x), int(y), int(w), int(h)),
                    "mouth_coords": None
                }

                if len(mouths) > 0:
                    mouths = sorted(mouths, key=lambda m: m[1])
                    mx, my, mw, mh = mouths[0]
                    mouth_coords = (int(x + mx), int(y + h // 2 + my), int(mw), int(mh))
                    data["mouth_coords"] = mouth_coords

                    # Save mouth crop
                    crop = frame[mouth_coords[1]:mouth_coords[1]+mh, mouth_coords[0]:mouth_coords[0]+mw]
                    crop_path = os.path.join(frame_save_dir, f"mouth_{saved_frame_count:04d}.png")
                    cv2.imwrite(crop_path, crop)
                    saved_frame_count += 1

                    print(f"🧑 Frame {frame_index}: Face + 👄 Mouth at {mouth_coords}")
                else:
                    print(f"⚠️  Frame {frame_index}: Face detected, but no mouth.")

                frame_metadata.append(data)

        frame_index += 1

    cap.release()
    print(f"\n📦 Done. Total frames: {frame_index}, Saved crops: {saved_frame_count}")
    return frame_metadata

# ========================================================
# ✅ STEP 5: RUN PREPROCESSING
# ========================================================
frame_metadata = process_video_frames(video_path, video_name)

# --- Section 6: Updated Frame Extraction with Correct Path ---
import os
import cv2
import json

# Absolute path to raw video
video_path = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data\raw_videos\vid_001.mpg"
video_name = os.path.splitext(os.path.basename(video_path))[0]

# Base output path
base_data_path = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data"
output_dir = os.path.join(base_data_path, video_name)

# Create directories
os.makedirs(os.path.join(output_dir, "frames"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "mouth_crops"), exist_ok=True)

metadata_path = os.path.join(output_dir, "metadata.json")

# Extract frames
cap = cv2.VideoCapture(video_path)
frame_idx = 0
metadata = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_filename = f"{frame_idx:04d}.jpg"
    frame_path = os.path.join(output_dir, "frames", frame_filename)
    cv2.imwrite(frame_path, frame)

    metadata.append({
        "frame": frame_filename,
        "timestamp": cap.get(cv2.CAP_PROP_POS_MSEC)
    })

    frame_idx += 1

cap.release()

# Save metadata
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Extracted {frame_idx} frames for video '{video_name}' and saved metadata.")
"""

In [1]:
# ========================================================
# 📘 VisoSpeak - 01_preprocessing.ipynb
# ========================================================
"""
This notebook performs the first stage of the lip reading pipeline:
Detects faces and mouths from a silent video and saves cropped mouth images and metadata.

🔹 INPUT:
    - A silent speaking video file (MPG format)
    - Located at: data/raw_videos/{video_name}.mpg

🔹 OUTPUT:
    - Cropped mouth images (PNG format)
        ➤ Saved to: data/frames/{video_name}/mouth_0000.png, ...
    - Metadata JSON containing mouth crop coordinates
        ➤ Saved to: data/processed/{video_name}/crop_metadata_{video_name}.json

🔹 MAIN STEPS:
    1. Load the video
    2. Detect faces and mouths using Haar cascades
    3. Save the first valid mouth crop and all mouth regions
    4. Export bounding box coordinates to JSON for downstream use
"""


'\nThis notebook performs the first stage of the lip reading pipeline:\nDetects faces and mouths from a silent video and saves cropped mouth images and metadata.\n\n🔹 INPUT:\n    - A silent speaking video file (MPG format)\n    - Located at: data/raw_videos/{video_name}.mpg\n\n🔹 OUTPUT:\n    - Cropped mouth images (PNG format)\n        ➤ Saved to: data/frames/{video_name}/mouth_0000.png, ...\n    - Metadata JSON containing mouth crop coordinates\n        ➤ Saved to: data/processed/{video_name}/crop_metadata_{video_name}.json\n\n🔹 MAIN STEPS:\n    1. Load the video\n    2. Detect faces and mouths using Haar cascades\n    3. Save the first valid mouth crop and all mouth regions\n    4. Export bounding box coordinates to JSON for downstream use\n'

In [1]:
# ========================================================
# ✅ STEP 1: IMPORTS & CONFIGURATION
# ========================================================
try:
    import cv2
    import matplotlib
    print(f"[✅] OpenCV {cv2.__version__}, Matplotlib {matplotlib.__version__} already installed.")
except ImportError:
    print("[⚠️] Installing missing dependencies...")
    !pip install opencv-python matplotlib
    import cv2
    import matplotlib


import os
import cv2
import json
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

# 📂 Absolute path to your project’s data folder
DATA_DIR = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data"

# Change this to your video name (without extension)
video_name = "bbaf2n"
video_file = f"{video_name}.mpg"
video_path = os.path.join(DATA_DIR, "raw_videos", video_file)

print(f"📥 Video Path: {video_path}")


[✅] OpenCV 4.11.0, Matplotlib 3.10.1 already installed.
📥 Video Path: C:\Users\PC\Desktop\phaseB\VisoSpeak\data\raw_videos\bbaf2n.mpg


In [2]:

# ========================================================
# ✅ STEP 2: DOWNLOAD HAAR CASCADES IF NEEDED
# ========================================================
def download_haar_cascades():
    print("\n🔧 Checking Haar cascade files...")
    haar_dir = cv2.data.haarcascades
    cascades = {
        "haarcascade_frontalface_default.xml":
        "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml",
        
        "haarcascade_mcs_mouth.xml":
        "https://raw.githubusercontent.com/atduskgreg/opencv-processing/master/lib/cascade-files/haarcascade_mcs_mouth.xml"
    }

    for file, url in cascades.items():
        path = os.path.join(haar_dir, file)
        if not os.path.exists(path):
            print(f"⬇️  Downloading {file}")
            urllib.request.urlretrieve(url, path)
        else:
            print(f"✅ {file} already exists.")

download_haar_cascades()


🔧 Checking Haar cascade files...
✅ haarcascade_frontalface_default.xml already exists.
✅ haarcascade_mcs_mouth.xml already exists.


In [3]:
# ========================================================
# ✅ STEP 3: LOAD HAAR CLASSIFIERS
# ========================================================
print("\n🧠 Loading Haar Classifiers...")
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
mouth_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_mcs_mouth.xml")

assert not face_cascade.empty(), "❌ Failed to load face cascade"
assert not mouth_cascade.empty(), "❌ Failed to load mouth cascade"
print("✅ Haar classifiers loaded.")




🧠 Loading Haar Classifiers...
✅ Haar classifiers loaded.


In [4]:
# ========================================================
# ✅ STEP 4: PROCESS VIDEO FRAMES
# ========================================================
def process_video_frames(video_path, video_name):
    print(f"\n🎬 Processing: {video_path}")
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"❌ Video not found: {video_path}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"❌ Could not open video file: {video_path}")

    frame_index = 0
    saved_frame_count = 0
    frame_metadata = []

    frame_save_dir = os.path.join(DATA_DIR, "frames", video_name)
    os.makedirs(frame_save_dir, exist_ok=True)

    while True:
        ret, frame = cap.read()
        if not ret:
            print("✅ End of video.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

        if len(faces) == 0:
            print(f"🔍 Frame {frame_index}: No face detected.")
        else:
            for (x, y, w, h) in faces:
                roi_gray = gray[y + h // 2 : y + h, x:x + w]
                mouths = mouth_cascade.detectMultiScale(roi_gray, scaleFactor=1.3, minNeighbors=20, minSize=(50, 30))

                data = {
                    "frame_index": frame_index,
                    "face_coords": (int(x), int(y), int(w), int(h)),
                    "mouth_coords": None
                }

                if len(mouths) > 0:
                    mouths = sorted(mouths, key=lambda m: m[1])
                    mx, my, mw, mh = mouths[0]
                    mouth_coords = (int(x + mx), int(y + h // 2 + my), int(mw), int(mh))
                    data["mouth_coords"] = mouth_coords

                    # Save mouth crop
                    crop = frame[mouth_coords[1]:mouth_coords[1]+mh, mouth_coords[0]:mouth_coords[0]+mw]
                    crop_path = os.path.join(frame_save_dir, f"mouth_{saved_frame_count:04d}.png")
                    cv2.imwrite(crop_path, crop)
                    saved_frame_count += 1

                    print(f"🧑 Frame {frame_index}: Face + 👄 Mouth at {mouth_coords}")
                else:
                    print(f"⚠️  Frame {frame_index}: Face detected, but no mouth.")

                frame_metadata.append(data)

        frame_index += 1

    cap.release()
    print(f"\n📦 Done. Total frames: {frame_index}, Saved crops: {saved_frame_count}")
    return frame_metadata


In [5]:
# ========================================================
# ✅ STEP 5: RUN PREPROCESSING
# ========================================================
frame_metadata = process_video_frames(video_path, video_name)




🎬 Processing: C:\Users\PC\Desktop\phaseB\VisoSpeak\data\raw_videos\bbaf2n.mpg


FileNotFoundError: ❌ Video not found: C:\Users\PC\Desktop\phaseB\VisoSpeak\data\raw_videos\bbaf2n.mpg

In [7]:
# --- Section 6: Updated Frame Extraction with Correct Path ---
import os
import cv2
import json

# Absolute path to raw video
video_path = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data\raw_videos\vid_001.mpg"
video_name = os.path.splitext(os.path.basename(video_path))[0]

# Base output path
base_data_path = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data"
output_dir = os.path.join(base_data_path, video_name)

# Create directories
os.makedirs(os.path.join(output_dir, "frames"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "mouth_crops"), exist_ok=True)

metadata_path = os.path.join(output_dir, "metadata.json")

# Extract frames
cap = cv2.VideoCapture(video_path)
frame_idx = 0
metadata = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_filename = f"{frame_idx:04d}.jpg"
    frame_path = os.path.join(output_dir, "frames", frame_filename)
    cv2.imwrite(frame_path, frame)

    metadata.append({
        "frame": frame_filename,
        "timestamp": cap.get(cv2.CAP_PROP_POS_MSEC)
    })

    frame_idx += 1

cap.release()

# Save metadata
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Extracted {frame_idx} frames for video '{video_name}' and saved metadata.")


✅ Extracted 75 frames for video 'vid_001' and saved metadata.
